# EDA — Home Credit application_train.csv

All the actual analysis lives in `src/eda.py` and `src/data_loader.py` — this notebook calls into it and shows the results. Full write-up: `reports/eda_summary.md`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import load_application_data, missing_value_report, target_balance
from src.eda import target_correlations

df = load_application_data()
df.shape

(307511, 122)

307,511 rows, 122 columns. One row is one loan application; `TARGET` is 1 if the client had a late payment on the loan in our sample.

In [2]:
target_balance(df)

repaid (0)       0.919271
defaulted (1)    0.080729
Name: proportion, dtype: float64

8.1% defaulted, an 11.4:1 imbalance. Accuracy is useless here — predicting "repaid" for everyone scores 91.9% while catching zero defaults. See `reports/figures/target_imbalance.png`.

In [3]:
missing = missing_value_report(df)
missing.head(10)

,missing_count,missing_pct
COMMONAREA_MEDI,214865,69.87
COMMONAREA_AVG,214865,69.87
COMMONAREA_MODE,214865,69.87
NONLIVINGAPARTMENTS_MEDI,213514,69.43
NONLIVINGAPARTMENTS_MODE,213514,69.43
NONLIVINGAPARTMENTS_AVG,213514,69.43
FONDKAPREMONT_MODE,210295,68.39
LIVINGAPARTMENTS_MODE,210199,68.35
LIVINGAPARTMENTS_MEDI,210199,68.35
LIVINGAPARTMENTS_AVG,210199,68.35


67 of 122 columns have missing values. The worst are apartment/building features (`COMMONAREA_MEDI` etc., ~70% missing) that only exist for applicants living in an apartment complex — missing is informative, not garbage. Full chart: `reports/figures/missingness.png`.

In [4]:
target_correlations(df).head(10)

EXT_SOURCE_3                  -0.178919
EXT_SOURCE_2                  -0.160472
EXT_SOURCE_1                  -0.155317
DAYS_BIRTH                     0.078239
REGION_RATING_CLIENT_W_CITY    0.060893
REGION_RATING_CLIENT           0.058899
DAYS_LAST_PHONE_CHANGE         0.055218
DAYS_ID_PUBLISH                0.051457
REG_CITY_NOT_WORK_CITY         0.050994
FLAG_EMP_PHONE                 0.045982
Name: TARGET, dtype: float64

The three `EXT_SOURCE_*` columns (external credit-bureau scores) dominate every other feature's correlation with `TARGET` by roughly 2x. Distributions of these plus income/credit/age: `reports/figures/key_feature_distributions.png`.

Full 5-finding write-up, including the `DAYS_EMPLOYED` sentinel-value bug (365243 = "not employed", not a real tenure), is in `reports/eda_summary.md`.